# Pretty Formula One

This notebook is used to fetch data from certain seasons and save them into files to be uploaded into hosted database

## Race Results

In [12]:
import fastf1
import json
import logging
logging.getLogger('fastf1').setLevel(logging.ERROR)

YEAR = 2024

with open(f"./data/drivers_{YEAR}.json", "r") as f:
    drivers_list = json.load(f)
    
schedule = fastf1.get_event_schedule(YEAR)
events = schedule["EventName"].to_list()
events = [event for event in events if "Testing" not in event]
races = [idx+1 for idx, _ in enumerate(events)]
    
all_rounds_data = []
for ROUND_ID in races:

    race_session = fastf1.get_session(YEAR, ROUND_ID, "R")
    race_session.load(telemetry=False, weather=False)

    race_points_map = {
        f"{row['DriverId']}_{YEAR}": int(row['Points']) 
        for _, row in race_session.results.iterrows()
    }

    id_to_number_map = {
        f"{row['DriverId']}_{YEAR}": row['DriverNumber'] 
        for _, row in race_session.results.iterrows()
    }

    sprint_points_map = {}
    has_sprint = False
    try:
        sprint_session = fastf1.get_session(YEAR, ROUND_ID, "S")
        sprint_session.load(laps=False, telemetry=False, weather=False)
        has_sprint = True
        sprint_points_map = {
            row['DriverNumber']: int(row['Points']) 
            for _, row in sprint_session.results.iterrows()
        }
    except Exception:
        pass

    race_results_list = []
    for driver in drivers_list:
        d_id = driver["id"]
        r_points = race_points_map.get(d_id, 0)
        s_points = 0
        if has_sprint:
            d_number = id_to_number_map.get(d_id)
            if d_number:
                s_points = sprint_points_map.get(d_number, 0)
                
        laps = race_session.laps
        tyre_data = laps[['Compound', 'LapNumber', 'DriverNumber', 'Stint']]
        lap_tyre_data = {}
        for _, row in tyre_data.iterrows():
            if row["Compound"] == "nan":
                continue
            driver_number = row["DriverNumber"]
            if driver_number not in lap_tyre_data:
                lap_tyre_data[driver_number] = []
            last_compound = lap_tyre_data[driver_number][-1]["compound"] if lap_tyre_data[driver_number] else None
            if row["Compound"] != last_compound:
                lap_tyre_data[driver_number].append({
                    "compound": row["Compound"],
                    "lapStart": row["LapNumber"],
                    "lapEnd": row["LapNumber"],
                    "stint": row["Stint"]
                })
            else:
                lap_tyre_data[driver_number][-1]["lapEnd"] = row["LapNumber"]

        race_results_list.append({
            "driver_id": d_id,
            "racePoints": r_points,
            "sprintPoints": s_points,
            "tyre_strat": lap_tyre_data.get(id_to_number_map.get(d_id), [])
        })

    event_info = race_session.event
    country = event_info["Country"]
    schedule = fastf1.get_event_schedule(YEAR)
    total_rounds = len(schedule[schedule["RoundNumber"] > 0])

    round_data = {
        "id": int(event_info["RoundNumber"]),
        "year": YEAR,
        "index": int(event_info["RoundNumber"]),
        "totalRounds": total_rounds,
        "name": event_info["EventName"],
        "nameVerbose": event_info["OfficialEventName"],
        "country": country,
        "backgroundImage": f"/assets/circuits/{country.lower().replace(' ', '_')}.png",
        "results": sorted(race_results_list, key=lambda x: x["racePoints"], reverse=True),
    }
    
    print(f"Processed round {ROUND_ID}: {event_info['EventName']}")

    all_rounds_data.append(round_data)
    
filename = f"./data/rounds_{YEAR}.json"

with open(filename, "w") as f:
    json.dump(all_rounds_data, f, separators=(',', ':'))

Processed round 1: Bahrain Grand Prix
Processed round 2: Saudi Arabian Grand Prix
Processed round 3: Australian Grand Prix
Processed round 4: Japanese Grand Prix
Processed round 5: Chinese Grand Prix
Processed round 6: Miami Grand Prix
Processed round 7: Emilia Romagna Grand Prix
Processed round 8: Monaco Grand Prix
Processed round 9: Canadian Grand Prix
Processed round 10: Spanish Grand Prix
Processed round 11: Austrian Grand Prix
Processed round 12: British Grand Prix
Processed round 13: Hungarian Grand Prix
Processed round 14: Belgian Grand Prix
Processed round 15: Dutch Grand Prix
Processed round 16: Italian Grand Prix
Processed round 17: Azerbaijan Grand Prix
Processed round 18: Singapore Grand Prix
Processed round 19: United States Grand Prix
Processed round 20: Mexico City Grand Prix
Processed round 21: São Paulo Grand Prix
Processed round 22: Las Vegas Grand Prix
Processed round 23: Qatar Grand Prix
Processed round 24: Abu Dhabi Grand Prix


## All Event Names and Weekend Dates

In [14]:
# getting all the name of the grand prix on the season
import fastf1
import json
from pprint import pprint
import logging
logging.getLogger('fastf1').setLevel(logging.ERROR)

YEAR = 2026
schedule = fastf1.get_event_schedule(YEAR)
events_dates = []
for event_name in schedule["EventName"].to_list():
    date = schedule[schedule["EventName"] == event_name]["Session1Date"].values[0]
    date = date.strftime("%Y-%m-%dT%H:%M:%S%z")
    events_dates.append({
        "name": event_name.replace("Grand Prix", "").strip(),
        "date": date,
        "season": YEAR,
    })
pprint(events_dates)
with open(f"./data/dates_{YEAR}.json", "w") as f:
    json.dump(events_dates, f, separators=(',', ':'))

[{'date': '2026-02-11T10:00:00+0300',
  'name': 'Pre-Season Testing',
  'season': 2026},
 {'date': '2026-02-11T10:00:00+0300',
  'name': 'Pre-Season Testing',
  'season': 2026},
 {'date': '2026-03-06T12:30:00+1100', 'name': 'Australian', 'season': 2026},
 {'date': '2026-03-13T11:30:00+0800', 'name': 'Chinese', 'season': 2026},
 {'date': '2026-03-27T11:30:00+0900', 'name': 'Japanese', 'season': 2026},
 {'date': '2026-04-10T14:30:00+0300', 'name': 'Bahrain', 'season': 2026},
 {'date': '2026-04-17T16:30:00+0300', 'name': 'Saudi Arabian', 'season': 2026},
 {'date': '2026-05-01T12:30:00-0400', 'name': 'Miami', 'season': 2026},
 {'date': '2026-05-22T12:30:00-0400', 'name': 'Canadian', 'season': 2026},
 {'date': '2026-06-05T13:30:00+0200', 'name': 'Monaco', 'season': 2026},
 {'date': '2026-06-12T13:30:00+0200', 'name': 'Barcelona', 'season': 2026},
 {'date': '2026-06-26T13:30:00+0200', 'name': 'Austrian', 'season': 2026},
 {'date': '2026-07-03T12:30:00+0100', 'name': 'British', 'season': 2026

## Drivers

In [10]:
import fastf1
import json
from pprint import pprint
import logging
logging.getLogger('fastf1').setLevel(logging.ERROR)

# change here the info that you want to fetch (this will result on the
# list of drivers that participated in the specified year)
YEAR = 2024

session = fastf1.get_session(YEAR, 'Monaco', 'R')
session.load(laps=False, telemetry=False, weather=False)
drivers_list = []
seen_drivers = set()
for _, row in session.results.iterrows():
    driver_slug = f"{row['DriverId']}_{YEAR}"
    if driver_slug not in seen_drivers:
        drivers_list.append({
            "id": driver_slug,
            "team": row["TeamName"],
            "abbreviation": row["Abbreviation"],
            "name": row["FullName"],
            "teamLogo": f"/assets/icons/{row['TeamName'].lower().replace(' ', '-')}.png",
        })
        seen_drivers.add(driver_slug)
        
with open(f"./data/drivers_{YEAR}.json", "w") as f:
    json.dump(drivers_list, f, separators=(',', ':'))